In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%cd ..

In [ ]:
import importlib
from omegaconf import OmegaConf
import torch
import numpy as np
from hp.core.evaluator import PriorEvaluator, IKEvaluator
from hp.core.cond_trainer import CondSO3PriorTrainer
from hp.analysis.eval_ik_tools import compute_geod_and_mpjpe_for_based_on_k_samples

In [ ]:
path_pretrained_models = "experiments/pretrained_models"

The presented results might slightly deviate from the paper since we moved from `pytorch3d` to `roma` for rotation conversions, which results in a different set of randomly generated rotations when sampling.

# Prior

In [ ]:
# Get the model
exp_id = 900
experiment_path = f"{path_pretrained_models}/id_{exp_id:04}"

config = OmegaConf.load(f"{experiment_path}/config.yaml")

config.device = "cuda:0"

config.only_eval = True
config.standard_mode = True
config.condition = False

module = importlib.import_module("hp.core.trainer")
class_ = getattr(module, config.trainer)
trainer = class_(config)
trainer.load_model(save_path=experiment_path)
    

In [ ]:
# Sample from the prior
with torch.no_grad():
    samples = trainer.sample(100)

In [ ]:
# Evaluate the model
d = PriorEvaluator.eval(config, trainer, render_human=False, plot_samples=False, save_plot=False, compute_d_wasser=False)
print(d)


# IK

In [ ]:
# Get the model

exp_id = 410
experiment_path = f"{path_pretrained_models}/id_{exp_id:04}"

config = OmegaConf.load(f"{experiment_path}/config.yaml")

config.device = "cuda:0"
config.batch_size = 2000
config.num_eval_samples = 10_000

np.random.seed(config.seed)
torch.manual_seed(config.seed)

config.standard_mode = True
config.only_eval = True
config.selected_joints = "None"
config.conditioning.vary_mask_prob = False

trainer = CondSO3PriorTrainer(config)
trainer.load_model()

seed = 10
np.random.seed(seed); torch.manual_seed(seed);


In [ ]:
# Evaluate the model
print("Avg LL: ", trainer.eval_estimator(num_samples=-1, complete=True))
d = IKEvaluator.eval_with_dataset(trainer, n_samples=5_000, dataset = "eval")

In [ ]:
dataset = "eval"; n_samples = 5_000
dataset = trainer.eval_dataset if dataset == "eval" else trainer.train_dataset
idx = list(np.random.permutation(len(dataset))[:n_samples])
dataset_samples = dataset[idx]
theta_raw, p3d = dataset_samples["theta"].to(trainer.config.device).reshape(n_samples,-1,3,3), dataset_samples["condition"].to(trainer.config.device)

In [ ]:
# Evaluate the model with various samples
trainer.config.conditioning.mask = False
n_samples = theta_raw.shape[0]
for k_samples in [1, 10]:
    ths, p3d_raw, gs, mjps = compute_geod_and_mpjpe_for_based_on_k_samples(n_samples, k_samples, p3d, theta_raw, 
                                                                           trainer=trainer, selected_joints=trainer.selected_joints)

# IK with occlusion

In [ ]:
exp_id = 384
experiment_path = f"{path_pretrained_models}/id_{exp_id:04}"

config = OmegaConf.load(f"{experiment_path}/config.yaml")

config.device = "cuda:0"
config.batch_size = 2000
config.num_eval_samples = 5_000

np.random.seed(config.seed)
torch.manual_seed(config.seed)

config.standard_mode = True 
config.only_eval = True
config.selected_joints = "None"
config.conditioning.vary_mask_prob = False

trainer = CondSO3PriorTrainer(config)
trainer.load_model()

seed = 10
np.random.seed(seed); torch.manual_seed(seed);


In [ ]:
trainer.config.conditioning.zero_mask_prob = 0.3
trainer.config.conditioning.vary_mask_prob = True
trainer.config.conditioning.mask_type = "random"
d = IKEvaluator.eval_with_dataset(trainer, n_samples=5000, dataset = "eval")
print(d)

In [ ]:
dataset = "eval"; n_samples = 5_000
dataset = trainer.eval_dataset if dataset == "eval" else trainer.train_dataset
idx = list(np.random.permutation(len(dataset))[:n_samples])
dataset_samples = dataset[idx]
theta_raw, p3d = dataset_samples["theta"].to(trainer.config.device).reshape(n_samples,-1,3,3), dataset_samples["condition"].to(trainer.config.device)

In [ ]:
# Randomly occluded
trainer.config.conditioning.zero_mask_prob = 0.3
trainer.config.conditioning.vary_mask_prob = True
trainer.config.conditioning.mask_type = "random"
n_samples = theta_raw.shape[0]
for k_samples in [1, 10]:
    ths, p3d_raw, gs, mjps = compute_geod_and_mpjpe_for_based_on_k_samples(n_samples, k_samples, p3d, theta_raw, 
                                                                           trainer=trainer, selected_joints=trainer.selected_joints)

In [ ]:
# Occluded joints
from hp.analysis.eval_ik_tools import compute_geod_and_mpjpe_for_based_on_k_samples

n_samples = 10_000
trainer.config.conditioning.mask = True
trainer.config.conditioning.vary_mask_prob = False
trainer.config.conditioning.mask_type = "selected_joints"

sel_joints_nomask = [""]
sel_joints_1 = ["LeftUpLeg", "LeftLeg", "LeftFoot"] #(1)
sel_joints_2 = ["LeftHand", "LeftArm", "LeftElbow"] #(2)
sel_joints_3 = ["RightShoulder", "RightArm"] # (3)
sel_joints_4 = ["LeftHand", "LeftArm", "LeftElbow", "RightToeBase", "RightFoot"]
sel_joints_smplik = ["LeftHand", "RightHand", "LeftFoot", "RightFoot", "Head"]

b_compute_mesh_error = False


for i_sel, sel_joints in enumerate([sel_joints_nomask, sel_joints_1, sel_joints_2, sel_joints_3, sel_joints_smplik]):
    
    print(f"=====================================")
    print(f"Selection {i_sel}: Eval with {n_samples} samples for mask_joints: {sel_joints}")
    
    for i_eval in range(1):
        print(f"== Evaluation {i_eval} ==")
        idx = list(np.random.permutation(len(dataset))[:n_samples])
        s = dataset[idx]
        theta_raw, p3d = s["theta"].to(trainer.config.device).reshape(n_samples,-1,3,3), s["condition"].to(trainer.config.device)
        trainer.config.conditioning.mask_joints = sel_joints
        n_samples = theta_raw.shape[0]
        for k_samples in [1,10]:
            ths, p3d_raw, gs, mjps, g_mean, mjpe_mean, p_mjpe_mean = compute_geod_and_mpjpe_for_based_on_k_samples(n_samples, k_samples, p3d, theta_raw, trainer=trainer, return_eval_values=True)

# 2D to 3D uplifting

In [ ]:
exp_id = 510
experiment_path = f"{path_pretrained_models}/id_{exp_id:04}"


config = OmegaConf.load(f"{experiment_path}/config.yaml")

config.device = "cuda:0"
config.batch_size = 2000
config.num_eval_samples = 5_000


np.random.seed(config.seed)
torch.manual_seed(config.seed)


config.standard_mode = True
config.only_eval = True
config.selected_joints = "None"
config.conditioning.vary_mask_prob = False

trainer = CondSO3PriorTrainer(config)
trainer.load_model()

seed = 10
np.random.seed(seed); torch.manual_seed(seed);


In [ ]:
# 1 sample
print("Avg LL: ", trainer.eval_estimator(num_samples=-1, complete=True))
d = IKEvaluator.eval_with_dataset(trainer, n_samples=5_000, dataset = "eval")
print(d)

In [ ]:
dataset = "eval"; n_samples = 5_000
dataset = trainer.eval_dataset if dataset == "eval" else trainer.train_dataset
idx = list(np.random.permutation(len(dataset))[:n_samples])
dataset_samples = dataset[idx]
theta_raw, p3d = dataset_samples["theta"].to(trainer.config.device).reshape(n_samples,-1,3,3), dataset_samples["condition"].to(trainer.config.device)

In [ ]:
trainer.config.conditioning.mask = False
n_samples = theta_raw.shape[0]
for k_samples in [1, 10]:
    ths, p3d_raw, gs, mjps = compute_geod_and_mpjpe_for_based_on_k_samples(n_samples, k_samples, p3d, theta_raw, 
                                                                           trainer=trainer, selected_joints=trainer.selected_joints)